# 02 — Agronomic Reference Table
**Project:** Supervised Classification of Agricultural Soil Types in Togo  
**Author:** Daniel ESSONANI | Supervised by [M. Leri TCHANTCHO](https://www.linkedin.com/in/leri-damigouri-tchantcho-28503873/)

---

## Overview

This notebook builds the **agronomic reference table** that maps each WRB soil type  
to agricultural characteristics:
- Fertility level
- Recommended crops
- Main constraints

This table is later joined to the cantons shapefile to enrich the interactive map popups.

---

## ⚠️ Known Issues & Solutions

| Issue | Cause | Solution |
|-------|-------|---------|
| French column names caused encoding issues in shapefile | ESRI Shapefile truncates column names to 10 chars and has encoding quirks | Renamed all columns to short ASCII English names before shapefile export |
| `Ferralsols` not in SoilGrids results | SoilGrids v2.0 uses updated WRB 2022 — `Ferralsols` renamed to `Acrisols`/`Lixisols` | Kept entry for compatibility but `Ferralsols` will likely never appear |


## 1. Import

In [ ]:
import pandas as pd
print("pandas loaded ✅")


## 2. Build the Reference Table

Based on FAO soil-crop suitability guidelines and West African agronomic literature.  
Each row corresponds to one WRB soil class found or expected in Togo.


In [ ]:
# ── Agronomic reference table ─────────────────────────────────────────────────
# Sources:
#   - FAO World Reference Base for Soil Resources (2022)
#   - ORSTOM pedological maps of Togo (1970-1980)
#   - West African soil-crop suitability literature

reference_data = [
    {
        "soil":        "Acrisols",
        "fertility":   "Moderate",
        "crops":       "Cassava, Pineapple, Oil palm, Rubber",
        "constraints": "High acidity, low natural fertility — lime application required"
    },
    {
        "soil":        "Luvisols",
        "fertility":   "Good",
        "crops":       "Maize, Sorghum, Cotton, Yam",
        "constraints": "Erosion risk on slopes"
    },
    {
        "soil":        "Gleysols",
        "fertility":   "Conditional",
        "crops":       "Rice (lowland), Market gardening",
        "constraints": "Waterlogged — drainage infrastructure required"
    },
    {
        "soil":        "Lixisols",
        "fertility":   "Good",
        "crops":       "Maize, Sorghum, Cotton, Groundnut",
        "constraints": "Low water retention capacity"
    },
    {
        "soil":        "Nitisols",
        "fertility":   "Very good",
        "crops":       "Coffee, Cocoa, Maize, Yam",
        "constraints": "Few constraints — among the best soils in Togo"
    },
    {
        "soil":        "Vertisols",
        "fertility":   "Moderate",
        "crops":       "Cotton, Sorghum",
        "constraints": "Very difficult to till — cracks when dry, sticky when wet"
    },
    {
        "soil":        "Cambisols",
        "fertility":   "Good",
        "crops":       "Maize, Sorghum, Legumes",
        "constraints": "Variable fertility depending on parent material"
    },
    {
        "soil":        "Ferralsols",
        "fertility":   "Low",
        "crops":       "Cassava, Pineapple",
        "constraints": "Very low natural fertility — heavy organic matter inputs needed"
    },
    {
        "soil":        "Fluvisols",
        "fertility":   "Very good",
        "crops":       "Rice, Market gardening, Maize",
        "constraints": "Flood risk during rainy season"
    },
    {
        "soil":        "Regosols",
        "fertility":   "Low",
        "crops":       "Millet, Groundnut",
        "constraints": "Very low water retention — drought-prone"
    },
    {
        "soil":        "Arenosols",
        "fertility":   "Low",
        "crops":       "Millet, Cowpea, Groundnut",
        "constraints": "Sandy — frequent moisture stress"
    },
    {
        "soil":        "Leptosols",
        "fertility":   "Very low",
        "crops":       "Pasture, Reforestation",
        "constraints": "Too shallow for most crops — rocky substrate"
    },
]

df_reference = pd.DataFrame(reference_data)

# Save
df_reference.to_csv("table_reference_sols.csv", index=False)
print(f"✅ Reference table saved: {len(df_reference)} soil types")
df_reference


## 3. Validate Coverage Against SoilGrids Results

In [ ]:
# ── Check that all soil types found in extraction are covered ─────────────────
try:
    df_sols = pd.read_csv("sols_togo_cantons.csv")
    soilgrids_types = set(df_sols["soil_dominant"].unique()) - {"ERROR"}
    reference_types = set(df_reference["soil"])

    missing = soilgrids_types - reference_types
    if missing:
        print(f"⚠️  Soil types in SoilGrids results NOT in reference table: {missing}")
        print("   → Add entries for these types above and re-run.")
    else:
        print("✅ All SoilGrids soil types are covered in the reference table.")

    extra = reference_types - soilgrids_types
    if extra:
        print(f"ℹ️  Soil types in reference table but NOT found in Togo: {extra}")
        print("   → These entries will not appear in the map but are kept for completeness.")

except FileNotFoundError:
    print("ℹ️  sols_togo_cantons.csv not found — run notebook 01 first to check coverage.")
